<a id="top"></a>
# PanSTARRS "20 queries" using MAST's TAP service: Queries with Cutout Visualizations


******

## Overview

This notebook is part of a series demonstrating how to address a set of scientific questions using SQL-like Astronomical Data Query Language (ADQL) queries via a Virtual Observatory standard Table Access Protocol (TAP) service at MAST. 

This series aims to be an introduction to how complex queries can be executed using TAP (which might otherwise not be possible to specify with `astroquery`), and to be a resource for how to access MAST databases after the MAST CASJobs service is retired.

These queries are drawn from the 20 queries for SDSS as presented by [Gray, Szalay, et al. (2002)](https://arxiv.org/abs/cs/0202014), adapted for the PanSTARRS PS1 database.

This notebook presents a subset of queries leveraging **filtering on column(s) using pre-specified criteria, together with cutouts for visualization of the resulting sample.**


For more examples of filtering techniques in TAP queries (including using bitmask filtering for data quality cuts), please see the [PS1 "20 queries" Basic Filtering tutorial]().

<!-- ../PS1_20q_filtering_tutorial/PS1_20q_basic_filtering_tutorial.html -->

<div class="alert alert-warning" style="color:red; background-color:#ffc5c5; border-color:red;">
<b>FIX LINK TO BASIC FILTERING NOTEBOOK ABOVE</b>
</div>


## Learning Goals
By the end of this tutorial, you will:

- Understand how to design and perform filtering queries (with joins) using TAP services, and obtain cutouts to display color images of sample objects.


****
### Table of Contents

* [Introduction](#introduction)
* [Imports](#imports)
* [Connect to TAP service](#connect-to-tap-service)
* [Obtaining information about PanSTARRS catalogs](#obtaining-information-about-panstarrs-catalogs)
* [Q4: Find galaxies with an isophotal surface brightness (SB) larger than 24 in the red band, with an ellipticity>0.5, and with the major axis of the ellipse between 30 and 60 arcsec (a large galaxy).](#q4)
  * [Constructing the query](#q4:-constructing-the-query)
  * [Inspecting & visualizing the results](#q4:-inspecting-&-visualizing-the-results)
* [Q5: Find all galaxies with a deVaucouleours profile (r$^{1/4}$ falloff of intensity on disk) and the photometric colors consistent with an elliptical galaxy.](#q5)
  * [Constructing the query](#q5:-constructing-the-query)
  * [Inspecting & visualizing the results](#q5:-inspecting-&-visualizing-the-results)
* [Conclusions](#conclusions)
* [Additional Resources](#additional-resources)
* [Citations](#citations)
* [About This Notebook](#about-this-notebook)

*******
## Introduction

Welcome! This notebook shows how to query MAST's PanSTARRS data using Astronomical Data Query Language (ADQL). We address two scientific questions by combining filtering constraints using ADQL "WHERE" clauses, and then obtaining RGB cutouts to visualize objects in our resulting samples. As you'll see in our examples, MAST uses a standard Table Access Protocol (TAP) to handle these queries.

These queries are drawn from (or closely modeled on) the "20 queries for SDSS" presented by [Gray, Szalay, et al. (2002)](https://arxiv.org/abs/cs/0202014). Taken as a whole, this collection provides worked examples on how to leverage relational database capabilities to answer scientific questions, with the aim of providing concrete starting points and references for designing queries for other research applications, both for PanSTARRS and other large data volume missions (including Roman).

<div class="alert alert-block alert-info">
<b>Note:</b> ADQL/SQL comments follow "--". Comments are used throughout the queries to explain the purpose of specific clauses.
</div>

## Imports
This tutorial makes use of the following libraries: 
- [*numpy*](https://numpy.org/) for numerical calculations
- [*pyvo*](https://pyvo.readthedocs.io) for querying the MAST catalogs via TAP
- [*matplotlib.pyplot*](https://matplotlib.org/stable/api/pyplot_summary.html#module-matplotlib.pyplot) for plotting data
- *time*, *datetime* to determine query duration
- *requests*, *warnings*, *BytesIO*, *PIL*, [*astropy.table Table*](https://docs.astropy.org/en/stable/table/index.html) to fetch & display cutout images of selected objects

In [ ]:
import numpy as np
import pyvo as vo
import matplotlib.pyplot as plt
import datetime
import time

import requests
import warnings
from io import BytesIO
from PIL import Image
from astropy.table import Table

--------
## Connect to TAP service

For all queries, we will be connecting to the PanSTARRS (PS1) Data release 2 (DR2) catalog. Specifically, we will use the `ps1_dr2` catalogs available from the new [postgres-backed TAP service](https://mast.stsci.edu/vo-tap/api/v0.1/mast_catalogs/), which offers improved performance relative to the legacy database (by factors of 100 or greater, in many cases).

See the [PS1 documentation](link_to_migration_guide_here) for information about the tables available with this new TAP service.
    
<div class="alert alert-warning" style="color:red; background-color:#ffc5c5; border-color:red;">
<b>FIX LINK TO MIGRATION GUIDE ABOVE</b>
</div>

We begin by connecting to the MAST PanSTARRS DR2 TAP service.

_(Note: See https://mast.stsci.edu/vo-tap/ for a full list of MAST TAP services.)_

In [ ]:
TAP_service = vo.dal.TAPService(
    "https://mast.stsci.edu/vo-tap/api/v0.1/mast_catalogs"
)

Using the pyvo `describe()` method, we can access an overview of the methods, capabilities, and maximum result set size for this service. 

In [ ]:
TAP_service.describe()

As expected, this service supports ADQL. You can also see the maximum result size is 100,000 rows.

## Obtaining information about PanSTARRS catalogs

In addition to the [PanSTARRS DR2 catalogs documentation](https://outerspace.stsci.edu/spaces/PANSTARRS/pages/298812351/PS1+Source+extraction+and+catalogs), 
we can search and explore information about the PS1 DR2 tables using **MAST's [Catalog Schema Browser](https://mast.stsci.edu/schema_browser)**. This schema browser provides searchable listings of column names, units, and descriptions for the PS1 DR2 tables available via this TAP service. 

We draw on this documentation and schema browser to identify the relevant tables & columns needed to address the queries below.


<div class="alert alert-warning" style="color:red; background-color:#ffc5c5; border-color:red;">
<i><b>**IF POSSIBLE:**</b></i>
<i>Integrete the schema browser.</i>
</div>

*********

## Q4

Query 4 poses: 

> **Find galaxies with an isophotal surface brightness (SB) larger than 24 in the red band, with an ellipticity>0.5, and with the major axis of the ellipse between 30 and 60 arcsec (a large galaxy).**


### Q4: Constructing the query

Here we will determine the surface brightness in the `r` band, using Kron magnitudes and radii. 
We will determine ellipticities and major axis sizes from Sersic fits.

These measurements are stored in the following [tables](link_to_migration_guide):

<div class="alert alert-warning" style="color:red; background-color:#ffc5c5; border-color:red;">
Link to migration guide in the above sentence
</div>

- `ps1_dr2.stack_object`: Kron radius `rkronrad` to get size for surface brightness calculation & size cuts, and `rkronmag` for surface brightness calculation
- `ps1_dr2.stack_model_fit_ser`: Use the minor/major axis ratio `rserab` to get the ellipticity (with $\mathrm{ellipiticy} = \sqrt{1-(b/a)^2}> 0.5$ translating to `rserab`=$b/a < \sqrt{0.75}$).

Additionally, we perform quality control by requiring `ps1_dr2.stack_object.ndetections>3` and `ps1_dr2.stack_object.nr>1` (number of detections total and in the `r` band), and `ps1_dr2.stack_object.rpsfqfperfect>0.9` to weed out some bad objects. We also require `ps1_dr2.stack_model_fit_ser.rserchisq>0` to ensure the Sersic fit has at least minimal quality.


Finally, to ensure the query does not exceed the time or maximum row limit for a single TAP query, we further restrict this query to PanSTARRS slice 16 (by requiring `objid` to be between 129500000000000000 and 133500000000000000).

We combine these constraints to construct the ADQL query below, with relevant columns and derived quantities ($r$-band surface magnitude, ellipticity) as our list of columns to be returned.

In [ ]:
adql_query = """
SELECT so.objid, so.ramean, so.decmean, so.rkronrad, so.rkronmag,
   so.rkronmag + 2.5*log10(PI()*POWER(so.rkronrad,2)) as rsurfmag,
   smf.rserab, smf.rserchisq,
   sqrt(1-power(smf.rserab,2)) as ellipticity
FROM ps1_dr2.stack_model_fit_ser AS smf
JOIN ps1_dr2.stack_object AS so ON 
    smf.objid=so.objid 
    AND smf.uniquepspsstid=so.uniquepspsstid
WHERE smf.objid BETWEEN 129500000000000000 AND 133500000000000000 -- select just slice 16
  AND so.objid BETWEEN 129500000000000000 AND 133500000000000000
  AND so.rkronrad BETWEEN 30 AND 60
  AND so.rkronmag > 0
  AND so.rkronmag + 2.5*log10(PI()*POWER(so.rkronrad,2)) < 24 -- mag per sq arcsec
  AND smf.rserchisq > 0
  AND smf.rserab < sqrt(0.75)
  AND so.rpsfqfperfect > 0.9
  AND so.ndetections > 3
  AND so.nr > 1
"""

We now submit this query to the TAP service.

In [ ]:
start = time.time()
job = TAP_service.run_async(adql_query)
end = time.time()
print(f"Elapsed time: {str(datetime.timedelta(seconds=end-start))}")

This query takes about 30 seconds to run and returns 67 rows. For all 32 slices, this would take a total time of ~24 minutes and return ~2100 rows.

### Q4: Inspecting & visualizing the results

To inspect our results, we convert the TAP results to an astropy table, for easy viewing.

In [ ]:
TAP_results = job.to_table()
TAP_results

To visualize the results, we retrieve cutouts for a subset of these galaxies. We'll need three functions for this:
1. `get_image_table`: Query for a list of images based on position
2. `get_imurl`: Query for URLs corresponding to the images in (1)
3. `get_im`: Actually retrieve the images we located in (2)

In [ ]:
# Adapted from https://spacetelescope.github.io/mast_notebooks/notebooks/PanSTARRS/PS1_image/PS1_image.html
def get_image_table(ra, dec, filters="grizy"):
    """
    Query ps1filenames.py service to get a list of images
    
    ra, dec = position in degrees
    filters = string with filters to include. includes all by default
    Returns a table with the results
    """
    service = "https://ps1images.stsci.edu/cgi-bin/ps1filenames.py"
    # The final URL appends our query to the PS1 image service
    url = f"{service}?ra={ra}&dec={dec}&filters={filters}"
    # Read the ASCII table returned by the url
    table = Table.read(url, format='ascii')
    return table


def get_imurl(ra, dec, size=240, output_size=None, filters="grizy", color=False):
    """
    Get URL for images in the table
    
    ra, dec = position in degrees
    size = extracted image size in pixels (0.25 arcsec/pixel)
    output_size = output (display) image size in pixels (default = size).
                  output_size has no effect for fits format images.
    filters = string with filters to include. choose from "grizy"
    color = if True, creates a color image (only for jpg or png format).
            Default is return a list of URLs for single-filter grayscale images.   
    Returns a string with the URL
    """
    im_format = "jpg"
        
    # Call the original helper function to get the table
    table = get_image_table(ra, dec, filters=filters)
    url = (f"https://ps1images.stsci.edu/cgi-bin/fitscut.cgi?"
           f"ra={ra}&dec={dec}&size={size}&format={im_format}")
    
    # Append an output size, if requested
    if output_size:
        url = url + f"&output_size={output_size}"
        
    # Sort filters from red to blue
    flist = ["yzirg".find(x) for x in table['filter']]
    table = table[np.argsort(flist)]
    
    if color:
        # We need at least 3 filters to create a color image
        if len(table) < 3:
            raise ValueError("at least three filters are required for an RGB color image")
        # If more than 3 filters, pick 3 filters from the availble results
        if len(table) > 3:
            table = table[[0, len(table)//2, len(table)-1]]
        # Create the red, green, and blue files for our image
        for i, param in enumerate(["red", "green", "blue"]):
            url = url + f"&{param}={table['filename'][i]}"
   
    else:
        # If not a color image, only one filter should be given.
        if len(table) > 1:
            warnings.warn('Too many filters for monochrome image. Using only 1st filter.')
        # Use red for monochrome images
        urlbase = url + "&red="
        url = []
        filename = table[0]['filename']
        url = urlbase+filename
    return url


def get_im(ra, dec, size=240, output_size=None, filters="g", color=False):
    """
    Get image at a sky position. Depends on get_imurl
    
    ra, dec = position in degrees
    size = extracted image size in pixels (0.25 arcsec/pixel)
    output_size = output (display) image size in pixels (default = size).
                  output_size has no effect for fits format images.
    filters = string with filters to include
    Returns the image
    """
    # Image URL
    url = get_imurl(
        ra, dec, size=size, filters=filters, 
        output_size=output_size, color=color
    )

    # JPEG: Request the file, read the bytes
    r = requests.get(url)
    im = Image.open(BytesIO(r.content))
    return im

We specify that all cutouts will be 2 arcmin on a side (converted to pixels), and show only cutouts for a dozen pre-selected set of objects (specified by their RA, Dec).

In [ ]:
# Set image size in pixels (0.25 arcsec/pix)
size = 480

# Get inds for a given pre-selected set of objects
inds = []
ras = [4.57722, 21.64157, 28.70062, 29.83149,
       37.35923, 46.70883, 56.91690, 117.46351,
       129.42904, 132.34119, 136.42660, 138.25536]
decs = [19.39308, 19.83059, 20.79907, 19.00765,
        20.21708, 18.68872, 18.80113, 18.82904,
        20.50383, 19.07499, 18.33797, 20.36527]

for ra, dec in zip(ras, decs):
    inds.append(
        int(np.where(
            (np.abs(TAP_results['ramean']-ra) < 0.00002)
            & (np.abs(TAP_results['decmean']-dec) < 0.00002)
        )[0][0])
    )

# Create the axes, and for each index, obtain and plot the cutout color image.
f, axes = plt.subplots(3, 4)
f.set_size_inches(16, 12)
axes = axes.flatten()
for i, ind in enumerate(inds):
    ra = TAP_results["ramean"][ind]
    dec = TAP_results["decmean"][ind]
    # Color image
    cim = get_im(ra, dec, size=size, filters="gri", color=True)
    
    # Color image subplot
    sgn = "+"
    if np.sign(dec) < 0:
        sgn = "-"
    axes[i].set_title(f'{ra:0.5f} {sgn}{dec:0.5f} (gri)')
    axes[i].imshow(cim, origin="upper")

    # 7 ticks:
    halfw = int(np.floor(size/2 * 0.25))
    ticklabels = np.linspace(-halfw, halfw, num=7, dtype=int)
    ticklocs = ticklabels / 0.25 + size/2
    axes[i].set_xticks(ticklocs, labels=ticklabels)
    axes[i].set_yticks(ticklocs[::-1], labels=ticklabels)

The dozen galaxies shown above are indeed mostly large galaxies (again, the cutouts are 2 arcmin on a side), as specified in our selection criteria.  However, some are bad images that we could likely have filtered out using quality flags. Specifically, 3 images show bright, saturated stars, while another is a poor quality image with numerous reduction artifacts.

-------------
## Q5


Query 5 poses: 

> **Find all galaxies with a deVaucouleours profile (r$^{1/4}$ falloff of intensity on disk) and the photometric colors consistent with an elliptical galaxy.**


### Q5: Constructing the query

This query, as implemented in Gray et al. for SDSS, is complex and difficult to replicate exactly (e.g., as the absorption/reddening tables are not available for PanSTARRS). Thus here we implement similar color cuts to make this example comparable to the SDSS query.

Furthermore, PS1 does not have likelihoods for the de Vaucouleurs versus Sersic morphological fits, so instead heuristic criteria are applied. If the Sersic fit failed, or if it has a higher chi-square value than the de Vaucouleurs fit, then we assume the galaxy is best fit by the de Vaucouleurs profile.

These measurements are available by combining the following [tables](link_to_migration_guide):

<div class="alert alert-warning" style="color:red; background-color:#ffc5c5; border-color:red;">
Link to migration guide in the above sentence
</div>

- `ps1_dr2.stack_object`: Kron magnitudes in `gri` for color & magnitude constraints (used in place of Petrosian magnitudes, as the PS1 documentation states Petrosian magnitudes are unreliable), Kron radius `rkronrad` to get size, Galactic latitude `b`, to avoid the Milky Way Galactic plane (where dust and foreground stars would impact measurements)
- `ps1_dr2.stack_model_fit_ser`: Sersic fits, including chi-square
- `ps1_dr2.stack_model_fit_de_v`: de Vaucouleurs fits, including chi-square


Additionally, we perform quality control by requiring `ps1_dr2.stack_object.ndetections>3` and `ps1_dr2.stack_object.nr>1` (number of detections total and in the `r` band; and the same in the `g` and `i` bands), and `ps1_dr2.stack_object.rpsfqfperfect>0.9` (and the same in the `g` and `i` bands) to weed out some bad objects. We also require `ps1_dr2.stack_model_fit_de_v.gdevchisq>0` (and similarly for `r` and `i`) to ensure the de Vaucouleurs fit has at least minimal quality. Finally, we use PSF and Kron magnitude differences to select for galaxies (excluding point sources).

Finally, to ensure the query does not exceed the time or maximum row limit for a single TAP query, we further restrict this query to PanSTARRS slice 16 (by requiring `objid` to be between 129500000000000000 and 133500000000000000).


The ADQL query below implements these constraints, after joining all three required tables together.

In [ ]:
adql_query = """
select so.objid, so.ramean, so.decmean, so.b,
   so.rkronrad, so.gkronmag, so.rkronmag, so.ikronmag,
   -- axis ratios and chi-square for de Vaucoulers fits
   fdev.gdevab, fdev.rdevab, fdev.idevab,
   fdev.gdevchisq, fdev.rdevchisq, fdev.idevchisq,
   -- axis ratios and chi-square for the Sersic fits
   fser.gserab, fser.rserab, fser.iserab,
   fser.gserchisq, fser.rserchisq, fser.iserchisq
from ps1_dr2.stack_model_fit_ser as fser
join ps1_dr2.stack_model_fit_de_v as fdev on fser.objid=fdev.objid and fser.uniquepspsstid=fdev.uniquepspsstid
join ps1_dr2.stack_object as so on fser.objid=so.objid and fser.uniquepspsstid=so.uniquepspsstid
where fser.objid between 129500000000000000 and 133500000000000000 -- select just slice 16
  and fdev.objid between 129500000000000000 and 133500000000000000
  and so.objid   between 129500000000000000 and 133500000000000000
  and abs(so.b) > 20                               -- avoid Milky Way plane
  and (so.gkronmag+so.rkronmag+so.ikronmag) > 0 -- require gri detections
  and (so.ipsfmag - so.ikronmag > 0.05)          -- extended sources (galaxies)
  and (so.rkronrad > 0)
  and (fdev.gdevchisq > 0)                         -- de Vaucoleurs fits in gri
  and (fdev.rdevchisq > 0)
  and (fdev.idevchisq > 0)
  -- require de Vaucoleurs model to be the best fit
  and ((fser.gserchisq < 0) or (fdev.gdevchisq < fser.gserchisq))
  and ((fser.rserchisq < 0) or (fdev.rdevchisq < fser.rserchisq))
  and ((fser.iserchisq < 0) or (fdev.idevchisq < fser.iserchisq))
  -- quality tests
  and so.gpsfqfperfect > 0.9
  and so.rpsfqfperfect > 0.9
  and so.ipsfqfperfect > 0.9
  and so.ndetections > 3
  and so.nr > 0
  and so.ng > 0
  and so.ni > 0
  -- replace petro mags with Kron mags (PS1 docs say Petro mags are not reliable)
  -- no reddening info in PS1, just set it to zero
  -- use Kron mags for the gri mags too
  -- Use Kron radius instead of petror50_r (probably not close, but too bad)
  and (so.ikronmag > 17.5)
  and (so.rkronmag > 15.5 OR so.rkronrad > 2)
  and (so.rkronmag < 30 and so.gkronmag < 30 and so.rkronmag < 30 and so.ikronmag < 30)
  and (so.rkronmag < 19.2)
  -- color constraint
  and ( ( (so.rkronmag < (13.1 + (7/3)*(so.gkronmag - so.rkronmag)
                        + 4 *(so.rkronmag - so.ikronmag) -4 * 0.18) )
          and ((so.rkronmag - so.ikronmag - (so.gkronmag - so.rkronmag)/4 - 0.18) BETWEEN -0.2 AND 0.2)
        )
      or
        ( (so.rkronmag < 19.5)
          and ((so.rkronmag - so.ikronmag - (so.gkronmag - so.rkronmag)/4 - 0.18)
               > (0.45 - 4*(so.gkronmag - so.rkronmag)))
          and ((so.gkronmag - so.rkronmag) > (1.35 + 0.25 *(so.rkronmag - so.ikronmag)))
        )
    )
"""

Again, we submit this query to the TAP service, and obtain our results.

In [ ]:
start = time.time()
job = TAP_service.run_async(adql_query)
end = time.time()
print(f"Elapsed time: {str(datetime.timedelta(seconds=end-start))}")

This query takes about 30 seconds to run and returned 527 rows. For all 32 slices, this would take a total time of ~24 minutes and return ~16,900 rows.

### Q5: Inspecting & visualizing the results

By now, this is likely familiar: let's convert our results to a table.

In [ ]:
TAP_results = job.to_table()
TAP_results

To visualize the results, again we retreive cutouts for a curated subset of a dozen of these galaxies (specified by their RA and $i$-band magnitudes).

Here, we will make each cutout a little more than 0.5 arcmin on a side (126 pixels), as we did not select for particularly large galaxies.

In [ ]:
inds = []

# Obtain the indices for the curated subset of galaxies
imags = [17.50, 17.78, 17.93, 18.05,
         18.15, 18.21, 18.29, 18.36, 
         18.41, 18.49, 18.55, 18.68]
ras = [20.14718, 357.07376, 49.39537, 214.34062,
       37.13989, 223.76736, 133.60074, 225.50517,
       60.74114, 116.48517, 207.24531, 9.84269]
for ra, imag in zip(ras, imags):
    inds.append(
        int(np.where(
            (np.abs(TAP_results['ikronmag']-imag) < 0.02)
            & (np.abs(TAP_results['ramean']-ra) < 0.00002)
        )[0][0])
    )
    
# Set image size 
size = 126 # a bit more than 0.5 arcmin

f, axes = plt.subplots(3, 4)
f.set_size_inches(16, 12)
axes = axes.flatten()
for i, ind in enumerate(inds):
    ra = TAP_results["ramean"][ind]
    dec = TAP_results["decmean"][ind]
    # Color image
    cim = get_im(ra, dec, size=size, filters="gri", color=True)
    
    # Color image subplot
    sgn = "+"
    if np.sign(dec) < 0:
        sgn = "-"
    axes[i].set_title(
        f'{ra:0.5f} {sgn}{dec:0.5f} i={TAP_results["ikronmag"][ind]:0.2f} (gri)'
    )
    axes[i].imshow(cim, origin="upper")

    # 7 ticks:
    halfw = int(np.floor(size/2 * 0.25))
    ticklabels = np.linspace(-halfw, halfw, num=7, dtype=int)
    ticklocs = ticklabels / 0.25 + size/2
    axes[i].set_xticks(ticklocs, labels=ticklabels)
    axes[i].set_yticks(ticklocs[::-1], labels=ticklabels)

This subset, shown above, is sorted by $i$-band magnitude (from bright to faint). Thanks to the multiple quality cuts included in our sample selection, we see all galaxies in this sample subset all appear to satisfy our criteria selecting for light profiles and colors consistent with elliptical galaxies.

------

## Conclusions

The queries above demonstrate the speed of MAST's new, more performant PS1 TAP service.  These specific queries are up to 8 times faster than is possible with MAST's CASJobs service. (Depending on the set of constraints specified, this service can have even greater speedups.)

See the full MAST [PanSTARRS tutorial list](../../panstarrs.md) for more tutorials demonstrating how to access PanSTARRS catalogs and other select "20 queries" examples.

----------

## Additional Resources

### Table Access Protocol

- IVOA standard for RESTful web service access to tabular data
- http://www.ivoa.net/documents/TAP/

### PanSTARRS 1 DR 2

- https://outerspace.stsci.edu/display/PanSTARRS/

### Astronomical Query Data Language (2.0)

- IVOA standard for querying astronomical data in tabular format, with geometric search support
- http://www.ivoa.net/documents/latest/ADQL.html

### PyVO

- an affiliated package for [astropy](https://www.astropy.org/)
- find and retrieve astronomical data available from archives that support standard IVOA virtual observatory service protocols.
- https://pyvo.readthedocs.io/en/latest/index.html


### Full list of MAST/TAP services
- A full list of available MAST TAP services can be found at:
- https://mast.stsci.edu/vo-tap


## Citations
If you use `astropy` for published research, please cite the
authors. Follow these links for more information about citing `astropy`:

* [Citing `astropy`](https://www.astropy.org/acknowledging.html)

If you use PanSTARRS data accessed through MAST for published research, 
please include the following acknowledgements, found at the following links:

* [Acknowledging PanSTARRS](https://archive.stsci.edu/publishing/mission-acknowledgements#section-895d38a0-86b3-4143-b521-6cc3312701f9)
* [Acknowledging MAST](https://archive.stsci.edu/gsc/mast_data_use.html)


## About this Notebook

**Authors**  Rick White, Sedona Price<br>
**Keywords:** Tutorial, TAP, pyvo, ADQL, PanSTARRS <br>
**Last Updated:** August 2026
***
[Top of Page](#top)
<img style="float: right;" src="https://raw.githubusercontent.com/spacetelescope/style-guides/master/guides/images/stsci-logo.png" alt="Space Telescope Logo" width="200px"/> 